# Demandes — projection 2050 — Norte Amazónica

Construit `Demands.csv` par cluster pour deux jeux de demande à l'horizon 2050, en généralisant
la méthode déjà validée en 2025 par `reality_access/demande_access.ipynb`
(A réseau + B/C bundle *sufficiency* au prorata RAMP, sans nouvelle simulation) sur l'année et la
trajectoire d'accès :

| Jeu | Trajectoire (`projections/output/*.csv`) | Utilisé par |
|---|---|---|
| **no_transition_late_access** | `acces_2050` (accès lent, universel en 2050) | scénarios `no_transition` et `late_access` — offre seule diffère (f_max=0 PV_UTILITY/BATT_LI en P0), la demande est identique |
| **early_access** | `acces_2035` (accès universel dès 2035) | scénario `early_access` |

| Composante | Provenance | Traitement |
|---|---|---|
| **A** — réseau | `projections/output/demande_source_A_projetee.csv` | énergie finale (convention AETN) → conversion en énergie utile via `Layers_in_out.csv`, comme en 2025 |
| **B** — ménages dispersés | RAMP **sufficiency** (`../sufficiency/data ramp/`), au prorata `hh_B(muni, 2050, trajectoire) / hh_total_muni_2024` | bundle *figé* depuis 2025 — RAMP n'est pas relancé, seul le nombre de ménages change |
| **C** — non desservis | — | demande nulle (households pas encore raccordés à cette date/trajectoire) |
| **Cooking** | recensement 2024, mis à l'échelle par la croissance du total des ménages (`menages_projetes.csv`) | indépendant du split A/B/C (Règle 4) |

Sortie : `output_energyscope_2050/{no_transition_late_access,early_access}/C{k}/Demands.csv` (jamais dans un chemin 2025).


## 0. Contrôle — `Layers_in_out.csv` vs référence EnergyScope

Le `Layers_in_out.csv` local (utilisé plus bas pour convertir l'énergie électrique RAMP/AETN en
énergie utile) doit correspondre exactement au fichier de référence 2025 `reality` — les
efficacités sont **figées** aux deux horizons de projection (jamais relâché, jamais neutralisé).


In [1]:
import pandas as pd

LIO_LOCAL_PATH = "../data/Layers_in_out.csv"
LIO_REFERENCE_PATH = "../../../EnergyScope_BO_nord_amazonia/Data/2025/reality/00_INDEP/Layers_in_out.csv"

lio_local = pd.read_csv(LIO_LOCAL_PATH, sep=";", header=0, index_col=0)
lio_reference = pd.read_csv(LIO_REFERENCE_PATH, sep=";", header=0, index_col=0)

problems = []

only_local_rows = sorted(set(lio_local.index) - set(lio_reference.index))
only_reference_rows = sorted(set(lio_reference.index) - set(lio_local.index))
if only_local_rows:
    problems.append(f"technologies only in local file: {only_local_rows}")
if only_reference_rows:
    problems.append(f"technologies only in EnergyScope reference: {only_reference_rows}")

only_local_cols = sorted(set(lio_local.columns) - set(lio_reference.columns))
only_reference_cols = sorted(set(lio_reference.columns) - set(lio_local.columns))
if only_local_cols:
    problems.append(f"layers only in local file: {only_local_cols}")
if only_reference_cols:
    problems.append(f"layers only in EnergyScope reference: {only_reference_cols}")

common_rows = sorted(set(lio_local.index) & set(lio_reference.index))
common_cols = sorted(set(lio_local.columns) & set(lio_reference.columns))
changed_rows = sorted(
    tech for tech in common_rows
    if not lio_local.loc[tech, common_cols].equals(lio_reference.loc[tech, common_cols])
)
if changed_rows:
    problems.append(f"technologies with different coefficients: {changed_rows}")

if problems:
    raise ValueError(
        f"Layers_in_out.csv differs from the EnergyScope reference ({LIO_REFERENCE_PATH}): "
        + "; ".join(problems)
    )

print(f"OK — Layers_in_out.csv matches the EnergyScope reference ({LIO_REFERENCE_PATH})")


OK — Layers_in_out.csv matches the EnergyScope reference (../../../EnergyScope_BO_nord_amazonia/Data/2025/reality/00_INDEP/Layers_in_out.csv)


## 1. Configuration — année, clusters, correspondances de noms, trajectoires

In [2]:
import os
import pandas as pd

YEAR = 2050
OUTPUT_DIR = f"output_energyscope_{YEAR}"

CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir",
        "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando",
        "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}
MUNI_TO_CLUSTER = {m: k for k, munis in CLUSTERS.items() for m in munis}
SECTOR_COLS = ["HOUSEHOLDS", "SERVICES", "INDUSTRY", "TRANSPORTATION",
               "PUBLIC_LIGHTING", "AGRICULTURE", "MINING", "FISHING_OTHERS"]

# Deux jeux de demande par horizon — P0 et P1 partagent le même jeu (no_transition_late_access) : la différence
# P0/P1 est côté offre (f_max=0 sur PV_UTILITY/BATT_LI en P0), hors scope de ce notebook.
TRAJ = {"no_transition_late_access": "acces_2050", "early_access": "acces_2035"}

# (municipio, departamento) du recensement/projections -> nom canonique RAMP. Deux "Santa Rosa"
# (Beni et Pando) désambiguées par département. Identique aux notebooks reality/reality_access.
CSVFINAL_TO_RAMP = {
    ("Ixiamas",               "La Paz"): "Ixiamas",
    ("Riberalta",             "Beni"):   "Riberalta",
    ("Guayaramerín",          "Beni"):   "Guayaramerín",
    ("Reyes",                 "Beni"):   "Reyes",
    ("Santa Rosa",            "Beni"):   "Santa_Rosa_Beni",
    ("Exaltación",            "Beni"):   "Exaltación",
    ("Cobija",                "Pando"):  "Cobija",
    ("Porvenir",              "Pando"):  "Porvenir",
    ("Bolpebra",              "Pando"):  "Bolpebra",
    ("Bella Flor",            "Pando"):  "Bella_Flor",
    ("Puerto Rico",           "Pando"):  "Puerto_Rico",
    ("San Pedro",             "Pando"):  "San_Pedro",
    ("Filadelfia",            "Pando"):  "Filadelfia",
    ("Puerto Gonzalo Moreno", "Pando"):  "Puerto_Gonzalo_Moreno",
    ("San Lorenzo",           "Pando"):  "San_Lorenzo",
    ("Sena",                  "Pando"):  "Sena",
    ("Santa Rosa",            "Pando"):  "Santa_Rosa_Pando",
    ("Ingavi",                "Pando"):  "Ingavi",
    ("Nueva Esperanza",       "Pando"):  "Nueva_Esperanza",
    ("Villa Nueva",           "Pando"):  "Villa_Nueva",
    ("Santos Mercado",        "Pando"):  "Santos_Mercado",
}

print(f"YEAR = {YEAR}  ->  OUTPUT_DIR = {OUTPUT_DIR}")
print(f"TRAJ = {TRAJ}")


YEAR = 2050  ->  OUTPUT_DIR = output_energyscope_2050
TRAJ = {'no_transition_late_access': 'acces_2050', 'early_access': 'acces_2035'}


## 2. Coefficients d'efficacité (`Layers_in_out.csv`)

RAMP et Source A sont en **électricité finale** ; EnergyScope attend l'**énergie utile de
service**. Mêmes coefficients que `reality`/`sufficiency`/`reality_access` (efficacités figées).

$$\text{GWh}_{\text{utile}} = \frac{\text{GWh}_{\text{électrique}}}{|\text{coeff}_{\text{ELECTRICITY}}|}$$


In [3]:
LAYER_TO_TECH = {
    'ELECTRICITY':                   None,
    'LIGHTING_R_C':                  'LED_BULB',
    'LIGHTING_P':                    'LED_LIGHT',
    'HEAT_HIGH_T':                   'IND_DIRECT_ELEC',
    'HEAT_LOW_T_SH':                 'DEC_DIRECT_ELEC',
    'HEAT_LOW_T_HW':                 'DEC_DIRECT_ELEC',
    'COOKING':                       'STOVE_ELEC',
    'PROCESS_COOLING':               'IND_ELEC_COLD',
    'SPACE_COOLING':                 'DEC_ELEC_COLD',
    'FOOD_PRESERVATION':             'REFRIGERATOR_EL',
    'MECHANICAL_ENERGY_COMM':        'COMM_MACHINERY_EL',
    'MECHANICAL_ENERGY_IND':         'IND_MACHINERY_EL',
    'MECHANICAL_ENERGY_MOV_AGR':     'TRACTOR_EL',
    'MECHANICAL_ENERGY_FIX_AGR':     'AGR_MACHINERY_EL',
    'MECHANICAL_ENERGY_MIN':         'MIN_MACHINERY_EL',
    'MECHANICAL_ENERGY_FISH_OTHERS': 'FISH_MACHINERY_EL',
    'NON_ENERGY':                    None,
}
MOBILITY_LAYERS = {'MOBILITY_PASSENGER', 'MOBILITY_FREIGHT', 'AVIATION_LONG_HAUL', 'SHIPPING'}

lio = pd.read_csv("../data/Layers_in_out.csv", sep=";", header=0, index_col=0)

LAYER_TO_COEFF = {
    layer: (1.0 if tech is None else abs(float(lio.loc[tech, "ELECTRICITY"])))
    for layer, tech in LAYER_TO_TECH.items()
}

def to_useful(gwh_elec, end_use):
    return gwh_elec / LAYER_TO_COEFF.get(end_use, 1.0)

print("Layer -> |ELECTRICITY coefficient|:")
for layer, coeff in LAYER_TO_COEFF.items():
    print(f"  {layer:<35} {coeff:.6f}")


Layer -> |ELECTRICITY coefficient|:
  ELECTRICITY                         1.000000
  LIGHTING_R_C                        2.941176
  LIGHTING_P                          2.941176
  HEAT_HIGH_T                         1.000000
  HEAT_LOW_T_SH                       1.000000
  HEAT_LOW_T_HW                       1.000000
  COOKING                             1.000000
  PROCESS_COOLING                     0.496500
  SPACE_COOLING                       0.400000
  FOOD_PRESERVATION                   2.792308
  MECHANICAL_ENERGY_COMM              1.212121
  MECHANICAL_ENERGY_IND               1.111111
  MECHANICAL_ENERGY_MOV_AGR           1.111111
  MECHANICAL_ENERGY_FIX_AGR           1.111111
  MECHANICAL_ENERGY_MIN               1.111111
  MECHANICAL_ENERGY_FISH_OTHERS       1.111111
  NON_ENERGY                          1.000000


## 3. Table `Demands` vide

21 lignes (une par couche d'usage final), tous secteurs à zéro. Format identique aux
`Demands.csv` de `reality`/`sufficiency`/`reality_access`.

In [4]:
def create_empty_demands():
    columns = ["Category", "Subcategory", "parameter name"] + SECTOR_COLS + ["Units"]
    rows = [
        ["Electricity", "Electricity",                            "ELECTRICITY",                   "[GWh]"],
        ["Lighting",    "Building lighting",                      "LIGHTING_R_C",                  "[GWh]"],
        ["Lighting",    "Public lighting",                        "LIGHTING_P",                    "[GWh]"],
        ["Heat",        "High temperature",                       "HEAT_HIGH_T",                   "[GWh]"],
        ["Heat",        "Space heating",                          "HEAT_LOW_T_SH",                 "[GWh]"],
        ["Heat",        "Hot water",                              "HEAT_LOW_T_HW",                 "[GWh]"],
        ["Heat",        "Cooking",                                "COOKING",                       "[GWh]"],
        ["Cold",        "Process cooling",                        "PROCESS_COOLING",               "[GWh]"],
        ["Cold",        "Space cooling",                          "SPACE_COOLING",                 "[GWh]"],
        ["Cold",        "Food preservation",                      "FOOD_PRESERVATION",             "[GWh]"],
        ["Mobility",    "Passenger",                              "MOBILITY_PASSENGER",            "[Mpkm]"],
        ["Mobility",    "Freight",                                "MOBILITY_FREIGHT",              "[Mtkm]"],
        ["Mobility",    "Long-haul passenger flights",            "AVIATION_LONG_HAUL",            "[Mpkm]"],
        ["Mobility",    "International shipping",                 "SHIPPING",                      "[Mtkm]"],
        ["Mechanical",  "Mechanical energy commercial",           "MECHANICAL_ENERGY_COMM",        "[GWh]"],
        ["Mechanical",  "Mechanical energy industrial",           "MECHANICAL_ENERGY_IND",         "[GWh]"],
        ["Mechanical",  "Mechanical energy agriculture mobility", "MECHANICAL_ENERGY_MOV_AGR",     "[GWh]"],
        ["Mechanical",  "Mechanical energy agriculture fixed",    "MECHANICAL_ENERGY_FIX_AGR",     "[GWh]"],
        ["Mechanical",  "Mechanical energy mining",               "MECHANICAL_ENERGY_MIN",         "[GWh]"],
        ["Mechanical",  "Mechanical energy fishing",              "MECHANICAL_ENERGY_FISH_OTHERS", "[GWh]"],
        ["Non-energy",  "Non-energy",                             "NON_ENERGY",                    "[GWh]"],
    ]
    df = pd.DataFrame(columns=columns)
    for i, row in enumerate(rows):
        df.loc[i] = [row[0], row[1], row[2]] + [0.0] * 8 + [row[3]]
    return df


def add_demands(df, contributions):
    '''Add {(sector, end_use): GWh} into a Demands table, in place.'''
    for (sector, end_use), gwh in contributions.items():
        if end_use in MOBILITY_LAYERS or sector not in SECTOR_COLS:
            continue
        df.loc[df["parameter name"] == end_use, sector] += gwh
    return df


## 4. Source A — demande réseau projetée

`projections/output/demande_source_A_projetee.csv` : sortie de `05_source_A_demande_projetee.ipynb`,
en **énergie finale** (convention AETN — contrôlé ci-dessous : le total région 2024 doit tomber
sur **180.10307 GWh**, pas 177.6, l'écart correspondant à l'exclusion erronée de San Lorenzo dans
le chiffre publié dans le mémoire). `SERVICES_OTHER` est fusionné dans `SERVICES`, comme en 2025.
Conversion en énergie utile via les mêmes coefficients (Section 2).


In [5]:
SOURCE_A_PATH = "../../projections/output/demande_source_A_projetee.csv"
source_a_raw = pd.read_csv(SOURCE_A_PATH)
source_a_raw["secteur"] = source_a_raw["secteur"].replace("SERVICES_OTHER", "SERVICES")

# Contrôle bloquant — le total Source A 2024 (baseline commune aux deux trajectoires) doit
# reproduire exactement le total mesuré AETN 2025 (180.10307 GWh final), pas le chiffre publié
# erroné de 177.6 GWh (exclusion de San Lorenzo).
EXPECTED_2024_TOTAL_GWH = 180.10307
for traj in TRAJ.values():
    total_2024 = source_a_raw.loc[
        (source_a_raw["annee"] == 2024) & (source_a_raw["trajectoire"] == traj), "demande_GWh"
    ].sum()
    print(f"Source A 2024 baseline ({traj}): {total_2024:.5f} GWh final")
    assert abs(total_2024 - EXPECTED_2024_TOTAL_GWH) < 1e-3, (
        f"Source A 2024 baseline = {total_2024:.5f} GWh, attendu {EXPECTED_2024_TOTAL_GWH} GWh "
        f"(180.10 GWh, PAS 177.6 — vérifier l'exclusion de San Lorenzo en amont)"
    )
print("OK — Source A 2024 baseline = 180.10307 GWh (San Lorenzo inclus)")


def source_a_by_cluster(traj):
    '''{cluster_id: {(sector, end_use): GWh utile}} for one trajectory, at YEAR.'''
    sub = source_a_raw[(source_a_raw["annee"] == YEAR) & (source_a_raw["trajectoire"] == traj)]
    result = {}
    for cluster_label, group in sub.groupby("cluster"):
        cluster_id = int(cluster_label.replace("C", ""))
        result[cluster_id] = {
            (sector, end_use): to_useful(g["demande_GWh"].sum(), end_use)
            for (sector, end_use), g in group.groupby(["secteur", "usage_final"])
        }
    return result


for jeu, traj in TRAJ.items():
    by_cluster = source_a_by_cluster(traj)
    print(f"\nSource A {jeu} ({traj}) @ {YEAR} — GWh utile par cluster:")
    for cluster_id in sorted(CLUSTERS):
        total = sum(by_cluster.get(cluster_id, {}).values())
        print(f"  C{cluster_id}: {total:8.4f} GWh utile")


Source A 2024 baseline (acces_2050): 180.10307 GWh final
Source A 2024 baseline (acces_2035): 180.10307 GWh final
OK — Source A 2024 baseline = 180.10307 GWh (San Lorenzo inclus)

Source A no_transition_late_access (acces_2050) @ 2050 — GWh utile par cluster:
  C1:  28.3935 GWh utile
  C2:   2.2373 GWh utile
  C3: 143.9620 GWh utile
  C4:  73.2050 GWh utile
  C5:  72.6937 GWh utile

Source A early_access (acces_2035) @ 2050 — GWh utile par cluster:
  C1:  28.3935 GWh utile
  C2:   2.2373 GWh utile
  C3: 143.9620 GWh utile
  C4:  73.2050 GWh utile
  C5:  72.6937 GWh utile


## 5. Ménages A/B/C projetés — `split_abc_projete.csv`

Sortie de `03_split_abc_projete.ipynb` : `hh_A`/`hh_B`/`hh_C` par municipalité, par trajectoire,
pour 2024/2035/2050. `hh_B` = ménages hors réseau ayant atteint le niveau *sufficiency*
(dispersés) ; `hh_C` = ménages pas encore desservis (demande nulle — `hh_C` doit être ≈0
seulement une fois l'accès universel atteint par la trajectoire considérée).


In [6]:
SPLIT_ABC_PATH = "../../projections/output/split_abc_projete.csv"
split_raw = pd.read_csv(SPLIT_ABC_PATH)

split_raw["muni_ramp"] = split_raw.apply(
    lambda r: CSVFINAL_TO_RAMP.get((str(r["municipio"]).strip(), str(r["departamento"]).strip())),
    axis=1,
)
unmapped = split_raw[split_raw["muni_ramp"].isna()][["municipio", "departamento"]].drop_duplicates()
if len(unmapped):
    raise ValueError(f"Unmapped municipalities in split_abc_projete.csv: {unmapped.values.tolist()}")

# Sanity check: the cluster column already present in the file must match CLUSTERS/MUNI_TO_CLUSTER.
mismatched = split_raw[
    split_raw["muni_ramp"].map(MUNI_TO_CLUSTER).apply(lambda k: f"C{k}") != split_raw["cluster"]
]
assert mismatched.empty, f"Cluster mismatch vs CLUSTERS for rows:\n{mismatched}"

# hh_total per municipality at the 2024 baseline (trajectory-independent — checked equal below).
hh_total_2024 = (
    split_raw[split_raw["annee"] == 2024]
    .drop_duplicates("muni_ramp")
    .set_index("muni_ramp")["hh_total"]
    .to_dict()
)

# hh_B lookup: (muni_ramp, trajectoire) -> hh_B at YEAR
hh_b_year = (
    split_raw[split_raw["annee"] == YEAR]
    .set_index(["muni_ramp", "trajectoire"])["hh_B"]
    .to_dict()
)
hh_c_year = (
    split_raw[split_raw["annee"] == YEAR]
    .set_index(["muni_ramp", "trajectoire"])["hh_C"]
    .to_dict()
)

print(f"hh_total_2024 (region): {sum(hh_total_2024.values()):,.0f}")
for jeu, traj in TRAJ.items():
    hh_b_tot = sum(hh_b_year.get((m, traj), 0.0) for m in MUNI_TO_CLUSTER)
    hh_c_tot = sum(hh_c_year.get((m, traj), 0.0) for m in MUNI_TO_CLUSTER)
    print(f"{jeu} ({traj}) @ {YEAR}: hh_B={hh_b_tot:,.1f}  hh_C={hh_c_tot:,.1f}"
          + ("  <- doit être ~0 (accès universel)" if abs(hh_c_tot) < 1.0 else ""))


hh_total_2024 (region): 84,209
no_transition_late_access (acces_2050) @ 2050: hh_B=5,481.5  hh_C=-0.0  <- doit être ~0 (accès universel)
early_access (acces_2035) @ 2050: hh_B=5,481.5  hh_C=-0.0  <- doit être ~0 (accès universel)


## 6. Source B — ménages dispersés, bundle *sufficiency* au prorata

**RAMP n'est pas relancé.** Pour chaque municipalité on relit la même courbe de charge annuelle
`sufficiency` (2025, inchangée) et on la met au prorata par

$$f_{\text{muni}}(\text{année}, \text{trajectoire}) = \frac{HH_B(\text{muni}, \text{année}, \text{trajectoire})}{HH_{\text{total,muni}}(2024)}$$

**`sufficiency_water_heating` est exclue** de cette proratisation (convention la plus récente et
validée, `analyse_GIS_phase2_projections/phase2_share_dispersion.ipynb` — les ménages
dispersés/hors réseau ne font pas d'eau chaude électrique en pratique) et remplacée par un terme
forfaitaire `ECS_APPOINT_KWH_HH` (fraction électrique réaliste de l'ECS *sufficiency*, 41.06 kWh/HH/an)
appliqué directement par ménage B (pas de proratisation par f_muni — déjà défini par ménage).


In [7]:
# Identique à reality_access/demande_access.ipynb, MOINS sufficiency_water_heating (exclue —
# voir markdown ci-dessus) qui est remplacée par le terme forfaitaire ECS_APPOINT_KWH_HH.
MAPPING_SUFF = {
    "sufficiency_illumination":            ("HOUSEHOLDS", "LIGHTING_R_C"),
    "sufficiency_ICT":                     ("HOUSEHOLDS", "ELECTRICITY"),
    "sufficiency_cold_storage":            ("HOUSEHOLDS", "FOOD_PRESERVATION"),
    "sufficiency_thermal_comfort":         ("HOUSEHOLDS", "SPACE_COOLING"),
    "big_school_illumination":             ("SERVICES",   "LIGHTING_R_C"),
    "big_school_ICT":                      ("SERVICES",   "ELECTRICITY"),
    "big_school_cold_storage":             ("SERVICES",   "FOOD_PRESERVATION"),
    "big_school_space_cooling":            ("SERVICES",   "SPACE_COOLING"),
    "health_center_illumination":          ("SERVICES",   "LIGHTING_R_C"),
    "health_center_ICT":                   ("SERVICES",   "ELECTRICITY"),
    "health_center_cold_storage":          ("SERVICES",   "FOOD_PRESERVATION"),
    "health_center_space_cooling":         ("SERVICES",   "SPACE_COOLING"),
    "health_center_water_heating":         ("SERVICES",   "HEAT_LOW_T_HW"),
    "health_center_water_supply":          ("SERVICES",   "ELECTRICITY"),
    "health_center_medical_equip":         ("SERVICES",   "ELECTRICITY"),
    "entertainment_business_illumination": ("SERVICES",   "LIGHTING_R_C"),
    "entertainment_business_ICT":          ("SERVICES",   "ELECTRICITY"),
    "entertainment_business_cold_storage": ("SERVICES",   "FOOD_PRESERVATION"),
    "restaurant_illumination":             ("SERVICES",   "LIGHTING_R_C"),
    "restaurant_cold_storage":             ("SERVICES",   "FOOD_PRESERVATION"),
    "restaurant_kitchen":                  ("SERVICES",   "COOKING"),
    "store_illumination":                  ("SERVICES",   "LIGHTING_R_C"),
    "store_ICT":                           ("SERVICES",   "ELECTRICITY"),
    "store_cold_storage":                  ("SERVICES",   "FOOD_PRESERVATION"),
    "workshop_illumination":               ("SERVICES",   "LIGHTING_R_C"),
    "workshop_ICT":                        ("SERVICES",   "ELECTRICITY"),
    "workshop_machinery":                  ("SERVICES",   "MECHANICAL_ENERGY_COMM"),
    "public_lighting_illumination":        ("PUBLIC_LIGHTING", "LIGHTING_P"),
    "rice_processing_rice_processing":     ("INDUSTRY",   "MECHANICAL_ENERGY_IND"),
}
ALLOWED_EXTRA_COLS = {"time", "sufficiency_water_heating"}

# ECS appoint électrique (analyse_GIS_phase2_projections/phase2_share_dispersion.ipynb):
# region-wide DEC_DIRECT_ELEC HEAT_LOW_T_DECEN production / total Hot water demand = 0.04825,
# appliqué à l'intensité ECS sufficiency (851 kWh/HH/an).
APPOINT_ELEC_SHARE = 0.04825
ECS_APPOINT_KWH_HH = 851 * APPOINT_ELEC_SHARE  # kWh/HH/an, électricité finale

SUFF_RAMP_DIR = "../sufficiency/data ramp"

_ramp_cache = {}

def _read_ramp_suff(muni):
    if muni not in _ramp_cache:
        path = f"{SUFF_RAMP_DIR}/{muni}/load_curve_energy_service_full_year_Norte_Amazonia.csv"
        if not os.path.exists(path):
            raise FileNotFoundError(f"RAMP sufficiency output missing for '{muni}': {path}")
        df = pd.read_csv(path)
        unmapped = [c for c in df.columns if c not in MAPPING_SUFF and c not in ALLOWED_EXTRA_COLS]
        if unmapped:
            raise ValueError(f"{muni}: unmapped RAMP columns would be silently dropped: {unmapped}")
        _ramp_cache[muni] = df
    return _ramp_cache[muni]


def source_b_by_cluster(traj):
    '''{cluster_id: {(sector, end_use): GWh utile}} for one trajectory, at YEAR.'''
    by_cluster = {k: {} for k in CLUSTERS}
    rows = []
    for muni, cluster_id in MUNI_TO_CLUSTER.items():
        hh_b = hh_b_year.get((muni, traj), 0.0)
        hh_tot_2024 = hh_total_2024[muni]
        f = hh_b / hh_tot_2024 if hh_tot_2024 > 0 else 0.0

        df = _read_ramp_suff(muni)
        contributions = by_cluster[cluster_id]
        for col, (sector, end_use) in MAPPING_SUFF.items():
            if col not in df.columns:
                continue
            gwh_elec = df[col].fillna(0).sum() / 60_000_000_000.0 * f
            key = (sector, end_use)
            contributions[key] = contributions.get(key, 0.0) + to_useful(gwh_elec, end_use)

        # ECS appoint — forfaitaire par ménage B, pas de proratisation par f (déjà par-ménage)
        ecs_gwh_elec = ECS_APPOINT_KWH_HH * hh_b / 1e6
        key = ("HOUSEHOLDS", "HEAT_LOW_T_HW")
        contributions[key] = contributions.get(key, 0.0) + to_useful(ecs_gwh_elec, "HEAT_LOW_T_HW")

        rows.append({"Cluster": cluster_id, "Municipio": muni, "hh_B": round(hh_b, 1), "f": round(f, 4)})

    return by_cluster, pd.DataFrame(rows)


for jeu, traj in TRAJ.items():
    by_cluster, detail = source_b_by_cluster(traj)
    print(f"\nSource B {jeu} ({traj}) @ {YEAR} — GWh utile par cluster:")
    for cluster_id in sorted(CLUSTERS):
        total = sum(by_cluster.get(cluster_id, {}).values())
        print(f"  C{cluster_id}: {total:8.4f} GWh utile")



Source B no_transition_late_access (acces_2050) @ 2050 — GWh utile par cluster:
  C1:   1.6109 GWh utile
  C2:   0.1577 GWh utile
  C3:   0.4136 GWh utile
  C4:   2.0779 GWh utile
  C5:   0.0000 GWh utile



Source B early_access (acces_2035) @ 2050 — GWh utile par cluster:
  C1:   1.6109 GWh utile
  C2:   0.1577 GWh utile
  C3:   0.4136 GWh utile
  C4:   2.0779 GWh utile
  C5:   0.0000 GWh utile


## 7. Cuisson — recensement 2024, mis à l'échelle par la croissance des ménages

Ménages comptés (2024, identique à `reality`/`reality_access`) :
`total_2024 − no_cocina_2024 − electricidad_2024` (cuisinières électriques exclues — déjà dans
Source A). Intensité : 1 344 kWh/ménage/an d'énergie **utile** (Pablo Jimenez Zabalaga, thèse
Roger Arias 2024–2025), inchangée aux deux horizons.

**Règle 4** — la cuisson suit le nombre de ménages qui cuisinent, repris du recensement, **pas**
du split A/B/C : le nombre de ménages 2024 est mis à l'échelle par la croissance du **total** des
ménages par municipalité (`projections/output/menages_projetes.csv`), indépendamment de la
composition A/B/C.


In [8]:
COL_DEPT, COL_MUNI = 1, 3
COL_COOK_TOTAL, COL_COOK_ELEC, COL_NO_COCINA = 46, 52, 54  # 2024 cooking section

csv_final = pd.read_csv("../../exctraction of data/output/CSV_final.csv", header=None)


def parse_int_cell(x):
    if pd.isna(x):
        return 0
    s = str(x).strip().replace("\xa0", "").replace(" ", "")
    if s in ("", "-", "nan"):
        return 0
    try:
        return int(float(s))
    except ValueError:
        return 0


cooking_hh_2024 = {}
for i in range(1, csv_final.shape[0]):
    key = (str(csv_final.iloc[i, COL_MUNI]).strip(), str(csv_final.iloc[i, COL_DEPT]).strip())
    if key not in CSVFINAL_TO_RAMP:
        continue
    total     = parse_int_cell(csv_final.iloc[i, COL_COOK_TOTAL])
    elec      = parse_int_cell(csv_final.iloc[i, COL_COOK_ELEC])
    no_cocina = parse_int_cell(csv_final.iloc[i, COL_NO_COCINA])
    cooking_hh_2024[CSVFINAL_TO_RAMP[key]] = total - no_cocina - elec

total_cooking_hh_2024 = sum(cooking_hh_2024.values())
print(f"Ménages cuisson non électrique 2024: {total_cooking_hh_2024:,} (attendu 82 328)")
assert total_cooking_hh_2024 == 82328, f"Mismatch: got {total_cooking_hh_2024}"

USEFUL_COOKING_PER_HH_GWh = 0.001344023  # GWh/household/year, inchangé aux deux horizons

# Croissance du total des ménages (indépendante du split A/B/C — Règle 4)
menages = pd.read_csv("../../projections/output/menages_projetes.csv")
menages["muni_ramp"] = menages.apply(
    lambda r: CSVFINAL_TO_RAMP.get((str(r["municipio"]).strip(), str(r["departamento"]).strip())),
    axis=1,
)
unmapped_m = menages[menages["muni_ramp"].isna()][["municipio", "departamento"]].drop_duplicates()
if len(unmapped_m):
    raise ValueError(f"Unmapped municipalities in menages_projetes.csv: {unmapped_m.values.tolist()}")

hh_total_year_col = f"hh_{YEAR}"
menages_idx = menages.set_index("muni_ramp")
hh_growth_factor = (menages_idx[hh_total_year_col] / menages_idx["hh_2024"]).to_dict()

cooking_hh_year = {m: cooking_hh_2024[m] * hh_growth_factor[m] for m in MUNI_TO_CLUSTER}

cooking_by_cluster = {}
print(f"\nCuisson non électrique @ {YEAR} (mise à l'échelle par croissance des ménages):")
for cluster_id in sorted(CLUSTERS):
    hh = sum(cooking_hh_year[m] for m in CLUSTERS[cluster_id])
    gwh = hh * USEFUL_COOKING_PER_HH_GWh
    cooking_by_cluster[cluster_id] = gwh
    print(f"  C{cluster_id}: {hh:>10,.1f} HH -> {gwh:7.4f} GWh utile")
print(f"  RÉGION: {sum(cooking_hh_year.values()):,.1f} HH "
      f"(2024: {total_cooking_hh_2024:,} HH, x{sum(cooking_hh_year.values())/total_cooking_hh_2024:.3f})")


Ménages cuisson non électrique 2024: 82,328 (attendu 82 328)

Cuisson non électrique @ 2050 (mise à l'échelle par croissance des ménages):
  C1:   14,551.9 HH -> 19.5580 GWh utile
  C2:    1,192.0 HH ->  1.6021 GWh utile
  C3:   54,531.9 HH -> 73.2922 GWh utile
  C4:   25,069.0 HH -> 33.6933 GWh utile
  C5:   22,166.5 HH -> 29.7923 GWh utile
  RÉGION: 117,511.3 HH (2024: 82,328 HH, x1.427)


## 8. Construction des `Demands.csv` — 2050

Par cluster, par jeu (no_transition_late_access, early_access) : Source A (réseau) + Source B (dispersés, bundle sufficiency)
+ Cooking (recensement, mis à l'échelle). Source C : demande nulle (pas de contribution).

In [9]:
demands_by_jeu = {}

for jeu, traj in TRAJ.items():
    sa_by_cluster = source_a_by_cluster(traj)
    sb_by_cluster, _ = source_b_by_cluster(traj)

    demands_by_jeu[jeu] = {}
    print(f"\n===== {jeu} ({traj}) @ {YEAR} =====")
    for cluster_id in sorted(CLUSTERS):
        df = create_empty_demands()
        add_demands(df, sa_by_cluster.get(cluster_id, {}))
        add_demands(df, sb_by_cluster.get(cluster_id, {}))
        df.loc[df["parameter name"] == "COOKING", "HOUSEHOLDS"] += cooking_by_cluster[cluster_id]

        out_path = f"{OUTPUT_DIR}/{jeu}/C{cluster_id}/Demands.csv"
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        df.to_csv(out_path, sep=";", index=False)
        demands_by_jeu[jeu][cluster_id] = df

        a_tot = sum(sa_by_cluster.get(cluster_id, {}).values())
        b_tot = sum(sb_by_cluster.get(cluster_id, {}).values())
        ck_tot = cooking_by_cluster[cluster_id]
        total = sum(df[s].astype(float).sum() for s in SECTOR_COLS)
        print(f"  C{cluster_id}: A={a_tot:7.4f}  B={b_tot:7.4f}  Cooking={ck_tot:6.4f}  "
              f"TOTAL={total:8.4f} GWh  (A={100*a_tot/total:4.1f}%  B={100*b_tot/total:4.1f}%  "
              f"Cooking={100*ck_tot/total:4.1f}%)  -> {out_path}")



===== no_transition_late_access (acces_2050) @ 2050 =====
  C1: A=28.3935  B= 1.6109  Cooking=19.5580  TOTAL= 49.5624 GWh  (A=57.3%  B= 3.3%  Cooking=39.5%)  -> output_energyscope_2050/no_transition_late_access/C1/Demands.csv
  C2: A= 2.2373  B= 0.1577  Cooking=1.6021  TOTAL=  3.9971 GWh  (A=56.0%  B= 3.9%  Cooking=40.1%)  -> output_energyscope_2050/no_transition_late_access/C2/Demands.csv
  C3: A=143.9620  B= 0.4136  Cooking=73.2922  TOTAL=217.6678 GWh  (A=66.1%  B= 0.2%  Cooking=33.7%)  -> output_energyscope_2050/no_transition_late_access/C3/Demands.csv
  C4: A=73.2050  B= 2.0779  Cooking=33.6933  TOTAL=108.9762 GWh  (A=67.2%  B= 1.9%  Cooking=30.9%)  -> output_energyscope_2050/no_transition_late_access/C4/Demands.csv


  C5: A=72.6937  B= 0.0000  Cooking=29.7923  TOTAL=102.4861 GWh  (A=70.9%  B= 0.0%  Cooking=29.1%)  -> output_energyscope_2050/no_transition_late_access/C5/Demands.csv



===== early_access (acces_2035) @ 2050 =====
  C1: A=28.3935  B= 1.6109  Cooking=19.5580  TOTAL= 49.5624 GWh  (A=57.3%  B= 3.3%  Cooking=39.5%)  -> output_energyscope_2050/early_access/C1/Demands.csv
  C2: A= 2.2373  B= 0.1577  Cooking=1.6021  TOTAL=  3.9971 GWh  (A=56.0%  B= 3.9%  Cooking=40.1%)  -> output_energyscope_2050/early_access/C2/Demands.csv


  C3: A=143.9620  B= 0.4136  Cooking=73.2922  TOTAL=217.6678 GWh  (A=66.1%  B= 0.2%  Cooking=33.7%)  -> output_energyscope_2050/early_access/C3/Demands.csv
  C4: A=73.2050  B= 2.0779  Cooking=33.6933  TOTAL=108.9762 GWh  (A=67.2%  B= 1.9%  Cooking=30.9%)  -> output_energyscope_2050/early_access/C4/Demands.csv
  C5: A=72.6937  B= 0.0000  Cooking=29.7923  TOTAL=102.4861 GWh  (A=70.9%  B= 0.0%  Cooking=29.1%)  -> output_energyscope_2050/early_access/C5/Demands.csv


## 9. Vérification — cohérence 2050

In [10]:
# À 2050, no_transition_late_access (acces_2050) et early_access (acces_2035) convergent : les deux trajectoires atteignent
# l'accès universel, la demande doit être IDENTIQUE cellule à cellule.
TOL = 1e-9
all_ok = True
for cluster_id in sorted(CLUSTERS):
    d1 = demands_by_jeu["no_transition_late_access"][cluster_id]
    d2 = demands_by_jeu["early_access"][cluster_id]
    for sec in SECTOR_COLS:
        diff = (d1[sec].astype(float) - d2[sec].astype(float)).abs()
        worst = diff.max()
        if worst > TOL:
            all_ok = False
            print(f"  ECART C{cluster_id} {sec}: max={worst:.2e}")
print("P1(2050) == early_access(2050) cellule à cellule:", "OK" if all_ok else "ECHEC")
assert all_ok, "no_transition_late_access et early_access devraient être identiques à 2050 (accès universel des deux côtés)"


P1(2050) == early_access(2050) cellule à cellule: OK


### Comparaison au total 2025

Baseline : `reality_access/output_energyscope/C{k}/Demands.csv` — même montage (A réseau + B/C
bundle *sufficiency* + cooking) que ce notebook, contrairement à `reality` qui garde le B
hors-réseau *réel* non-bundle. Comparaison structurellement cohérente (pommes avec pommes).

In [11]:
BASELINE_2025_DIR = "../reality_access/output_energyscope"

rows = []
for jeu in TRAJ:
    for cluster_id in sorted(CLUSTERS):
        d2025 = pd.read_csv(f"{BASELINE_2025_DIR}/C{cluster_id}/Demands.csv", sep=";")
        total_2025 = sum(d2025[s].astype(float).sum() for s in SECTOR_COLS)
        total_year = sum(demands_by_jeu[jeu][cluster_id][s].astype(float).sum() for s in SECTOR_COLS)
        rows.append({"Jeu": jeu, "Cluster": f"C{cluster_id}", "GWh_2025": total_2025,
                     f"GWh_{YEAR}": total_year, "facteur_croissance": total_year / total_2025})

comp = pd.DataFrame(rows)
print(comp.to_string(index=False, formatters={
    "GWh_2025": "{:8.4f}".format, f"GWh_{YEAR}": "{:8.4f}".format,
    "facteur_croissance": "{:.3f}x".format}))

for jeu in TRAJ:
    sub = comp[comp["Jeu"] == jeu]
    reg_2025 = sub["GWh_2025"].sum()
    reg_year = sub[f"GWh_{YEAR}"].sum()
    print(f"\n{jeu} RÉGION: {reg_2025:.4f} GWh (2025) -> {reg_year:.4f} GWh ({YEAR})  "
          f"x{reg_year/reg_2025:.3f}")


  Jeu Cluster GWh_2025 GWh_2050 facteur_croissance
no_transition_late_access      C1  33.7072  49.5624             1.470x
no_transition_late_access      C2   2.3661   3.9971             1.689x
no_transition_late_access      C3 153.9374 217.6678             1.414x
no_transition_late_access      C4  58.3931 108.9762             1.866x
no_transition_late_access      C5  76.5492 102.4861             1.339x
   early_access      C1  33.7072  49.5624             1.470x
   early_access      C2   2.3661   3.9971             1.689x
   early_access      C3 153.9374 217.6678             1.414x
   early_access      C4  58.3931 108.9762             1.866x
   early_access      C5  76.5492 102.4861             1.339x

no_transition_late_access RÉGION: 324.9531 GWh (2025) -> 482.6896 GWh (2050)  x1.485

early_access RÉGION: 324.9531 GWh (2025) -> 482.6896 GWh (2050)  x1.485


### Contrôle croisé — Source B vs `cluster_summary.csv` (GIS breakeven)

Comparaison d'ordre de grandeur, **pas une égalité stricte** : `demande_dispersee_GWh` /
`menages_cluster` de `analyse_GIS_phase2_projections/output/<année>/cluster_summary.csv`
classe les ménages "dispersés" par distance au réseau (breakeven GIS), alors que Source B ici
vient du split censitaire A/B/C (`hh_B`) — deux populations proches mais non identiques. Le
recoupement n'est pertinent qu'au point où chaque trajectoire atteint l'accès universel
(early_access à 2035 ; no_transition_late_access et early_access tous deux à 2050).

In [12]:
GIS_SUMMARY_PATH = f"../../analyse_GIS_phase2_projections/output/{YEAR}/cluster_summary.csv"
if os.path.exists(GIS_SUMMARY_PATH):
    gis = pd.read_csv(GIS_SUMMARY_PATH)
    gis = gis[gis["Cluster"].isin([f"C{k}" for k in CLUSTERS])]
    print(f"GIS cluster_summary ({YEAR}) — demande_dispersee_GWh (référence distance-based):")
    for jeu, traj in TRAJ.items():
        sb_by_cluster, _ = source_b_by_cluster(traj)
        print(f"\n  Jeu {jeu} ({traj}):")
        for _, row in gis.iterrows():
            cluster_id = int(row["Cluster"].replace("C", ""))
            our_b_gwh = sum(sb_by_cluster.get(cluster_id, {}).values())
            gis_gwh = row["demande_dispersee_GWh"]
            print(f"    {row['Cluster']}: notebook B={our_b_gwh:7.4f} GWh utile   "
                  f"GIS demande_dispersee={gis_gwh:7.4f} GWh (final, autre classification)")
else:
    print(f"(pas de {GIS_SUMMARY_PATH} — contrôle croisé sauté)")


GIS cluster_summary (2050) — demande_dispersee_GWh (référence distance-based):



  Jeu no_transition_late_access (acces_2050):
    C1: notebook B= 1.6109 GWh utile   GIS demande_dispersee= 2.0786 GWh (final, autre classification)
    C2: notebook B= 0.1577 GWh utile   GIS demande_dispersee= 0.1321 GWh (final, autre classification)
    C3: notebook B= 0.4136 GWh utile   GIS demande_dispersee= 0.6478 GWh (final, autre classification)
    C4: notebook B= 2.0779 GWh utile   GIS demande_dispersee= 1.5431 GWh (final, autre classification)
    C5: notebook B= 0.0000 GWh utile   GIS demande_dispersee= 0.0000 GWh (final, autre classification)



  Jeu early_access (acces_2035):
    C1: notebook B= 1.6109 GWh utile   GIS demande_dispersee= 2.0786 GWh (final, autre classification)
    C2: notebook B= 0.1577 GWh utile   GIS demande_dispersee= 0.1321 GWh (final, autre classification)
    C3: notebook B= 0.4136 GWh utile   GIS demande_dispersee= 0.6478 GWh (final, autre classification)
    C4: notebook B= 2.0779 GWh utile   GIS demande_dispersee= 1.5431 GWh (final, autre classification)
    C5: notebook B= 0.0000 GWh utile   GIS demande_dispersee= 0.0000 GWh (final, autre classification)
